In [ ]:
import random
import json
from datetime import datetime
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import pycountry

from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, WANING_COMPARISON, WANING_TIMEOUTS, WANING_RERUNS, MOB_SOURCE_ABBREVS
from emu_renewal.outputs import get_param_vals_by_analysis
from emu_renewal.utils import get_analysis_paths, get_countries_by_continent
from emu_renewal.plotting import plot_waning_comparison_spagh, plot_waning_comparison_proc_disp

In [ ]:
from emu_renewal.constants import ANALYSIS_NAMES

In [ ]:
all_countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
waning_paths = get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries)

In [ ]:
analysis_paths
sample_countries = random.sample(list(analysis_paths), n_samples)
sample_analyses = [[iso3, random.sample(list(analysis_paths[iso3]), 1)[0]] for iso3 in sample_countries]

\newpage
## Variable process median values
The following plots show the median value of the non-mechanistic variable process
with and without the incorporation of waning immunity into the model.
Twelve country-analysis type combinations were randomly selected without replacement
for this analysis.

\vspace{1.5cm}

\newpage
## Dispersion parameter posterior distributions
The following plots show the posterior distribution of the dispersion to the variable process
with and without the incorporation of waning immunity into the model.
The same analyses are compared as in the previous plot.
\vspace{1.5cm}

In [ ]:
first_waning_paths = dict(list(waning_paths.items())[:12])

In [ ]:
countries_by_cont = get_countries_by_continent(all_countries)

In [ ]:
from emu_renewal.plotting import get_standard_subplot

In [ ]:
fig, axes = get_standard_subplot(n_samples, 4)
flat_axes = axes.ravel()
param = "dispersion_proc"
for c, (iso3, mob_type) in enumerate(sample_analyses):

    # Gather the paths together
    sample_path = waning_paths[iso3]
    analysis_path = analysis_paths[iso3]
    run_paths = {"waning": sample_path, "no_waning": analysis_path}

    # Get the posterior values with and without waning
    posts = [get_param_vals_by_analysis(param, p)[mob_type] for p in run_paths.values()]
    combined_disps = pd.concat(posts, axis=1)
    combined_disps.columns = run_paths.keys()

    # Plot the posterior comparison
    ax = flat_axes[c]
    sns.kdeplot(combined_disps, ax=ax, fill=True, alpha=0.1, linewidth=1.5, common_norm=False)
    ax.set_title(f"{pycountry.countries.lookup(iso3).name}, {MOB_SOURCE_ABBREVS[mob_type]}")
    ax.set_yticks([])
    ax.set_ylabel("")

fig.tight_layout()